# R1 · Nombres et calcul — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r1_nombres_calcul/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r1_nombres_calcul/03_notebook.ipynb`, créée par `python tools/start_chapter.py R1`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R1.1 à R1.14, R1.17 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R1.21 | PGCD par l'algorithme d'Euclide et fraction irréductible · `gcd`, `simplify_fraction` | 🔨 | ★★ | 40 |
|  | R1.22 | Additionner deux fractions sans passer par les décimaux · `add_fractions` | 🔨 | ★★ | 30 |
|  | R1.25 | Calculer exactement avec `fractions.Fraction` | 📦 | ★ | 10 |
|  | R1.28 | Chasse à l'erreur : cinq calculs de fractions et de puissances | 🐛 | ★ | 15 |
|  | R1.29 | Chasse à l'erreur : `-3**2` et les parenthèses | 🐛 | ★ | 10 |
| B | R1.23 | La notation scientifique à n chiffres significatifs · `scientific_notation` | 🔨 | ★★ | 40 |
|  | R1.26 | Prédire : arrondir deux fois, est-ce arrondir une fois ? | 🔮 | ★ | 10 |
|  | R1.30 | Les ordres de grandeur sur une échelle : de l'atome aux paramètres d'un LLM | 🎨 | ★ | 15 |
| C | R1.24 | Évolutions successives et moyenne pondérée · `successive_changes`, `weighted_mean` | 🔨 | ★ | 30 |
|  | R1.27 | Après n hausses de 10 % : tracer et comparer à une hausse de 10n % | 🔬 | ★★ | 20 |
|  | R1.31 | Écrire la docstring d'une fonction (format NumPy) | 🛠️ | ★ | 15 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R1")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : priorités, fractions, puissances, racines, notation scientifique, pourcentages, moyennes.
- Écrire les six fonctions de `mymath/arith.py` et les tester.
- Calculer exactement avec `Fraction`, repérer les erreurs de calcul classiques, voir une hausse répétée s'envoler.

**Rappel express.** $\frac{a}{b} + \frac{c}{d} = \frac{ad + cb}{bd}$ ; diviser, c'est multiplier par l'inverse ; $a^{-n} = \frac{1}{a^n}$ ; $\sqrt{ab} = \sqrt{a}\sqrt{b}$ ; une évolution de $t$ multiplie par $1 + t$, et les coefficients successifs se multiplient.

## Partie 0 · Vérifier tes exercices papier (✏️ R1.1 à R1.14, 🔗 R1.17)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire (une fraction entre guillemets, `"11/20"` ; un décimal avec un point, `0.75` ; une liste, `[3.5, -2]` ; un texte entre guillemets, `"XYZ"`), jamais le calcul lui-même. Les réponses pas encore remplies affichent ⏳.

**Ex R1.1 — Placer des nombres sur la droite numérique et les ranger**

In [ ]:
# Ex R1.1: the values you found on paper
answer_R1_1a = ...  # a) the six letters in increasing order, as text
answer_R1_1b = ...  # b) an integer
answer_R1_1c = ...  # c) the letter(s), as text
answer_R1_1d = ...  # d) the letter(s), as text

for letter, answer in zip("abcd", [answer_R1_1a, answer_R1_1b, answer_R1_1c, answer_R1_1d]):
    wbm.check(f"R1.1{letter}", answer)

**Ex R1.2 — Calculer avec les priorités et les signes**

In [ ]:
# Ex R1.2: the values you found on paper
answer_R1_2a = ...  # a) an integer
answer_R1_2b = ...  # b) an integer
answer_R1_2c = ...  # c) an integer
answer_R1_2d = ...  # d) an integer
answer_R1_2e = ...  # e) an integer
answer_R1_2f = ...  # f) an integer
answer_R1_2g = ...  # g) an integer
answer_R1_2h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R1_2a, answer_R1_2b, answer_R1_2c, answer_R1_2d, answer_R1_2e, answer_R1_2f, answer_R1_2g, answer_R1_2h]):
    wbm.check(f"R1.2{letter}", answer)

**Ex R1.3 — Calcul mental avec la distributivité : 17 × 99, 23 × 101**

In [ ]:
# Ex R1.3: the values you found on paper
answer_R1_3a = ...  # a) an integer
answer_R1_3b = ...  # b) an integer
answer_R1_3c = ...  # c) an integer
answer_R1_3d = ...  # d) an integer
answer_R1_3e = ...  # e) an integer

for letter, answer in zip("abcde", [answer_R1_3a, answer_R1_3b, answer_R1_3c, answer_R1_3d, answer_R1_3e]):
    wbm.check(f"R1.3{letter}", answer)

**Ex R1.4 — Simplifier des fractions avec le PGCD**

In [ ]:
# Ex R1.4: the values you found on paper
answer_R1_4a = ...  # a) an integer
answer_R1_4b = ...  # b) an irreducible fraction, as text
answer_R1_4c = ...  # c) an integer
answer_R1_4d = ...  # d) an irreducible fraction, as text
answer_R1_4e = ...  # e) an irreducible fraction, as text
answer_R1_4f = ...  # f) an irreducible fraction, as text

for letter, answer in zip("abcdef", [answer_R1_4a, answer_R1_4b, answer_R1_4c, answer_R1_4d, answer_R1_4e, answer_R1_4f]):
    wbm.check(f"R1.4{letter}", answer)

**Ex R1.5 — Additionner, multiplier et diviser des fractions**

In [ ]:
# Ex R1.5: the values you found on paper
answer_R1_5a = ...  # a) an irreducible fraction, as text
answer_R1_5b = ...  # b) an irreducible fraction, as text
answer_R1_5c = ...  # c) an irreducible fraction, as text
answer_R1_5d = ...  # d) an irreducible fraction, as text
answer_R1_5e = ...  # e) an irreducible fraction, as text
answer_R1_5f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_R1_5a, answer_R1_5b, answer_R1_5c, answer_R1_5d, answer_R1_5e, answer_R1_5f]):
    wbm.check(f"R1.5{letter}", answer)

**Ex R1.6 — Comparer des fractions**

In [ ]:
# Ex R1.6: the values you found on paper
answer_R1_6a = ...  # a) a fraction, as text
answer_R1_6b = ...  # b) the three letters in increasing order, as text
answer_R1_6c = ...  # c) a fraction, as text
answer_R1_6d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_R1_6a, answer_R1_6b, answer_R1_6c, answer_R1_6d]):
    wbm.check(f"R1.6{letter}", answer)

**Ex R1.7 — Appliquer les règles des puissances**

In [ ]:
# Ex R1.7: the values you found on paper
answer_R1_7a = ...  # a) an integer
answer_R1_7b = ...  # b) an integer
answer_R1_7c = ...  # c) a fraction, as text
answer_R1_7d = ...  # d) an integer
answer_R1_7e = ...  # e) a fraction, as text
answer_R1_7f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_R1_7a, answer_R1_7b, answer_R1_7c, answer_R1_7d, answer_R1_7e, answer_R1_7f]):
    wbm.check(f"R1.7{letter}", answer)

**Ex R1.8 — Puissances négatives et puissances de 10**

In [ ]:
# Ex R1.8: the values you found on paper
answer_R1_8a = ...  # a) a fraction, as text
answer_R1_8b = ...  # b) an integer (or a fraction as text)
answer_R1_8c = ...  # c) a decimal
answer_R1_8d = ...  # d) the list [m, k]
answer_R1_8e = ...  # e) an integer
answer_R1_8f = ...  # f) an integer
answer_R1_8g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_R1_8a, answer_R1_8b, answer_R1_8c, answer_R1_8d, answer_R1_8e, answer_R1_8f, answer_R1_8g]):
    wbm.check(f"R1.8{letter}", answer)

**Ex R1.9 — Simplifier des racines carrées**

In [ ]:
# Ex R1.9: the values you found on paper
answer_R1_9a = ...  # a) an integer
answer_R1_9b = ...  # b) an integer
answer_R1_9c = ...  # c) an integer
answer_R1_9d = ...  # d) an integer
answer_R1_9e = ...  # e) a decimal
answer_R1_9f = ...  # f) an integer
answer_R1_9g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_R1_9a, answer_R1_9b, answer_R1_9c, answer_R1_9d, answer_R1_9e, answer_R1_9f, answer_R1_9g]):
    wbm.check(f"R1.9{letter}", answer)

**Ex R1.10 — Écrire en notation scientifique et arrondir**

In [ ]:
# Ex R1.10: the values you found on paper
answer_R1_10a = ...  # a) the list [m, k]
answer_R1_10b = ...  # b) the list [m, k]
answer_R1_10c = ...  # c) an integer
answer_R1_10d = ...  # d) a decimal
answer_R1_10e = ...  # e) an integer
answer_R1_10f = ...  # f) the list [m, k]

for letter, answer in zip("abcdef", [answer_R1_10a, answer_R1_10b, answer_R1_10c, answer_R1_10d, answer_R1_10e, answer_R1_10f]):
    wbm.check(f"R1.10{letter}", answer)

**Ex R1.11 — Partie entière, arrondi au plus proche, arrondi supérieur**

In [ ]:
# Ex R1.11: the values you found on paper
answer_R1_11a = ...  # a) an integer
answer_R1_11b = ...  # b) an integer
answer_R1_11c = ...  # c) an integer
answer_R1_11d = ...  # d) an integer
answer_R1_11e = ...  # e) an integer
answer_R1_11f = ...  # f) an integer
answer_R1_11g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_R1_11a, answer_R1_11b, answer_R1_11c, answer_R1_11d, answer_R1_11e, answer_R1_11f, answer_R1_11g]):
    wbm.check(f"R1.11{letter}", answer)

**Ex R1.12 — Évolutions successives et évolution réciproque**

In [ ]:
# Ex R1.12: the values you found on paper
answer_R1_12a = ...  # a) a number
answer_R1_12b = ...  # b) a percentage
answer_R1_12c = ...  # c) a percentage
answer_R1_12d = ...  # d) a percentage
answer_R1_12e = ...  # e) a percentage
answer_R1_12f = ...  # f) a percentage

for letter, answer in zip("abcdef", [answer_R1_12a, answer_R1_12b, answer_R1_12c, answer_R1_12d, answer_R1_12e, answer_R1_12f]):
    wbm.check(f"R1.12{letter}", answer)

**Ex R1.13 — Règle de trois et proportionnalité**

In [ ]:
# Ex R1.13: the values you found on paper
answer_R1_13a = ...  # a) a number
answer_R1_13b = ...  # b) a number
answer_R1_13c = ...  # c) a number
answer_R1_13d = ...  # d) a number
answer_R1_13e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_R1_13a, answer_R1_13b, answer_R1_13c, answer_R1_13d, answer_R1_13e]):
    wbm.check(f"R1.13{letter}", answer)

**Ex R1.14 — Moyenne pondérée : la note finale et la somme pondérée d'un neurone**

In [ ]:
# Ex R1.14: the values you found on paper
answer_R1_14a = ...  # a) a decimal
answer_R1_14b = ...  # b) a decimal
answer_R1_14c = ...  # c) a number
answer_R1_14d = ...  # d) a decimal

for letter, answer in zip("abcd", [answer_R1_14a, answer_R1_14b, answer_R1_14c, answer_R1_14d]):
    wbm.check(f"R1.14{letter}", answer)

**Ex R1.17 — Combien de paramètres, combien d'octets ? Un réseau en notation scientifique**

In [ ]:
# Ex R1.17: the values you found on paper
answer_R1_17a = ...  # a) an integer
answer_R1_17b = ...  # b) an integer
answer_R1_17c = ...  # c) an integer
answer_R1_17d = ...  # d) the list [m, k]
answer_R1_17e = ...  # e) an integer

for letter, answer in zip("abcde", [answer_R1_17a, answer_R1_17b, answer_R1_17c, answer_R1_17d, answer_R1_17e]):
    wbm.check(f"R1.17{letter}", answer)

## Partie A · Calculer exactement : PGCD, fractions, priorités

Fiche §2 à §4. Tes deux premières fonctions de `mymath` (le PGCD et l'addition de fractions), le calcul exact avec `Fraction`, et deux chasses à l'erreur.

In [ ]:
# Tools of the notebook (parts A to C)
import doctest
import math
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import run_pytest_group

### Ex R1.21 — PGCD par l'algorithme d'Euclide et fraction irréductible · `gcd`, `simplify_fraction` 🔨 ★★ ⏱️ 40 min
**Objectif :** programmer l'algorithme d'Euclide, puis s'en servir pour rendre une fraction irréductible.  
**Prérequis :** fiche §3 (l'algorithme d'Euclide) ; workbookIA 0A §100.4.2 (boucle `while`) et §100.6.2 (`raise`) · **mymath :** `arith.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en R0.9). Ouvre ta copie `mon_travail/mymath/arith.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. N'utilise ni `math.gcd` ni le module `fractions` : les tests s'en servent pour vérifier tes résultats.

- a) Écris `gcd(a, b)` : tant que `b` n'est pas nul, remplace le couple `(a, b)` par `(b, a % b)` ; à la fin, `a` est le PGCD. Travaille avec les valeurs sans signe (`abs`), et vérifie à la main sur le tableau de la fiche (1071 et 462).
- b) Écris `simplify_fraction(numerator, denominator)` : divise les deux par leur PGCD (avec `//`, pour rester en entiers), puis mets le signe au numérateur ; une `ValueError` si le dénominateur est nul.

La vérification calcule `gcd(2310, 1092)` et `simplify_fraction(-1092, 2310)`, puis lance les tests.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R1")  # reload your saved file

In [ ]:
with wbm.attempt("R1.21a"):
    wbm.check("R1.21a", mymath.arith.gcd(2310, 1092), computed=True)
with wbm.attempt("R1.21b"):
    wbm.check("R1.21b", list(mymath.arith.simplify_fraction(-1092, 2310)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r1_arith.py", "test_gcd_ or test_simplify_fraction_ or test_arith_uses")

### Ex R1.22 — Additionner deux fractions sans passer par les décimaux · `add_fractions` 🔨 ★★ ⏱️ 30 min
**Objectif :** additionner deux fractions exactement, avec des entiers seulement.  
**Prérequis :** R1.21 ; fiche §3 (additionner) · **mymath :** `arith.py` · **Parcours :** C, E

Écris `add_fractions(a, b)` dans `mon_travail/mymath/arith.py` : `a` et `b` sont des couples `(numérateur, dénominateur)`. Mets-les au même dénominateur (le produit des dénominateurs suffit), additionne les numérateurs, puis rends le résultat irréductible avec **ta** fonction `simplify_fraction` (R1.21). Une `ValueError` si un dénominateur est nul.

Aucun flottant : pas de `/`, seulement `*`, `+` et `//`. La vérification calcule `add_fractions((7, 15), (-3, 10))`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R1")  # reload your saved file

In [ ]:
with wbm.attempt("R1.22"):
    wbm.check("R1.22", list(mymath.arith.add_fractions((7, 15), (-3, 10))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r1_arith.py", "test_add_fractions_")

### Ex R1.25 — Calculer exactement avec `fractions.Fraction` 📦 ★ ⏱️ 10 min
**Objectif :** calculer une somme et un produit exactement avec `Fraction`, et voir ce que les flottants en font.  
**Prérequis :** fiche §3, « Calculer exactement avec Python » · **Parcours :** C, E

Avec `Fraction` (déjà importé dans la cellule des outils), calcule :
- a) la somme $S = 1 + \dfrac{1}{2} + \dfrac{1}{3} + \dots + \dfrac{1}{10}$, exactement ;
- b) le produit $P = \left(1 - \dfrac{1}{2}\right)\left(1 - \dfrac{1}{3}\right) \cdots \left(1 - \dfrac{1}{10}\right)$, exactement ;
- c) le même produit $P$ calculé en **flottants** (avec `1 - 1/k`) : vaut-il exactement `0.1` (`== 0.1`) ?

Une boucle `for k in range(...)` fait l'affaire (0A §100.4.2). Pour b), écris les premiers facteurs sur papier : devines-tu le résultat avant de l'exécuter ?

In [ ]:
answer_R1_25a = ...  # a) a Fraction (computed with a loop)
answer_R1_25b = ...  # b) a Fraction
answer_R1_25c = ...  # c) True or False (computed with floats)

In [ ]:
for letter, answer in zip("abc", [answer_R1_25a, answer_R1_25b, answer_R1_25c]):
    wbm.check(f"R1.25{letter}", answer)

### Ex R1.28 — Chasse à l'erreur : cinq calculs de fractions et de puissances 🐛 ★ ⏱️ 15 min
**Objectif :** repérer les erreurs classiques de calcul sur les fractions et les puissances, et les corriger.  
**Prérequis :** fiche §3 et §4 · **Parcours :** C, E

Un camarade a fait cinq calculs. **Sans calculatrice**, trouve ceux qui sont faux.

| | Calcul |
|---|---|
| a | $\dfrac{3}{4} + \dfrac{2}{7} = \dfrac{5}{11}$ |
| b | $\dfrac{3}{4} \times \dfrac{8}{9} = \dfrac{2}{3}$ |
| c | $5^3 \times 5^2 = 25^5$ |
| d | $\left(\dfrac{2}{5}\right)^{-1} = \dfrac{5}{2}$ |
| e | $\dfrac{10}{3} \div \dfrac{5}{6} = \dfrac{50}{18}$ |

- a) Donne les lettres des calculs faux, dans un texte : par exemple `"bd"`.
- b) Le bon résultat du calcul e (un entier).
- c) Le bon résultat du calcul a (une fraction irréductible, entre guillemets).

In [ ]:
answer_R1_28a = ...  # a) the letters of the wrong lines, as text, e.g. "bd"
answer_R1_28b = ...  # b) an integer
answer_R1_28c = ...  # c) an irreducible fraction, as text

In [ ]:
for letter, answer in zip("abc", [answer_R1_28a, answer_R1_28b, answer_R1_28c]):
    wbm.check(f"R1.28{letter}", answer)

### Ex R1.29 — Chasse à l'erreur : `-3**2` et les parenthèses 🐛 ★ ⏱️ 10 min
**Objectif :** repérer les formules que Python calcule autrement que prévu, faute de parenthèses.  
**Prérequis :** fiche §2 ; R0 §2 · **Parcours :** C, E

Chaque ligne veut calculer la formule écrite en commentaire, pour un nombre `x` déjà défini. **Sans rien exécuter**, trouve les lignes qui calculent **autre chose** que leur formule (pour certaines valeurs de `x`).

```python
a = -x**2            # ligne 1 :  (−x)²
b = (x + 1) / 2 * x  # ligne 2 :  (x + 1)x / 2
c = 2 ** -1          # ligne 3 :  1/2
d = x / 2 * 3        # ligne 4 :  x / 6
e = (-2) ** 4        # ligne 5 :  (−2)⁴
f = 10 - (x - 4)     # ligne 6 :  10 − x + 4
g = 1 / 2 + x        # ligne 7 :  1 / (2 + x)
```

- a) Donne la liste des numéros des lignes fausses, par exemple `[1, 2]`.
- b) Écris la fonction `line4` corrigée, qui calcule vraiment $\dfrac{x}{6}$ ; la vérification calcule `line4(3)`.

In [ ]:
answer_R1_29a = ...  # a) the list of the faulty lines, e.g. [1, 2]


def line4(x):
    """Line 4, corrected: return x / 6."""
    raise NotImplementedError("écris la formule corrigée")

In [ ]:
wbm.check("R1.29a", answer_R1_29a)
with wbm.attempt("R1.29b"):
    wbm.check("R1.29b", line4(3), computed=True)

## Partie B · Notation scientifique, arrondis et ordres de grandeur

Fiche §6. La notation scientifique programmée, un arrondi qui surprend, et une échelle des ordres de grandeur.

### Ex R1.23 — La notation scientifique à n chiffres significatifs · `scientific_notation` 🔨 ★★ ⏱️ 40 min
**Objectif :** programmer le passage d'un nombre à sa notation scientifique arrondie, cas limites compris.  
**Prérequis :** fiche §6 · **mymath :** `arith.py` · **Parcours :** C

Écris `scientific_notation(x, digits=3)` dans `mon_travail/mymath/arith.py` : elle renvoie le couple `(mantisse, exposant)` avec $1 \le |\text{mantisse}| < 10$, la mantisse arrondie à `digits` chiffres significatifs (`round(m, digits - 1)`).

Une démarche, à suivre dans la docstring :
1. `x == 0` est un cas à part (docstring) ;
2. l'exposant est l'entier $k$ tel que $10^k \le |x| < 10^{k+1}$. `math.log10(y)` donne le nombre (pas forcément entier) $e$ tel que $10^e = y$ : `math.log10(1000)` vaut 3, `math.log10(2000)` environ 3,3 (R6 l'étudiera). Son plancher, `math.floor(math.log10(abs(x)))`, donne donc presque toujours $k$ (vérifie que la mantisse obtenue est bien entre 1 et 10, et corrige d'un rang sinon) ;
3. la mantisse est $x / 10^k$, puis on l'arrondit ;
4. **le piège** : l'arrondi peut donner 10 (9,996 à trois chiffres) ; on écrit alors 1,00 et l'exposant augmente de 1.

La vérification calcule `scientific_notation(299792458, digits=4)` et `scientific_notation(-0.0009996)`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R1")  # reload your saved file

In [ ]:
with wbm.attempt("R1.23a"):
    wbm.check("R1.23a", list(mymath.arith.scientific_notation(299792458, digits=4)), computed=True)
with wbm.attempt("R1.23b"):
    wbm.check("R1.23b", list(mymath.arith.scientific_notation(-0.0009996)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r1_arith.py", "test_scientific_notation_")

### Ex R1.26 — Prédire : arrondir deux fois, est-ce arrondir une fois ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir le résultat de deux arrondis successifs, et le comparer à un arrondi direct.  
**Prérequis :** fiche §6, « Chiffres significatifs et arrondis » · **Parcours :** C

`round(x, 1)` arrondit au dixième, `round(x)` à l'unité. **Avant d'exécuter quoi que ce soit**, prédis :
- a) `round(round(3.46, 1))` : on arrondit d'abord 3,46 au dixième, puis ce résultat à l'unité ;
- b) `round(3.46)` : on arrondit directement à l'unité.

Écris tes deux prédictions (des entiers), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R1_26a = ...  # a) an integer, written BEFORE running the experiment below
prediction_R1_26b = ...  # b) an integer

In [ ]:
wbm.check("R1.26a", prediction_R1_26a)
wbm.check("R1.26b", prediction_R1_26b)

**L'expérience.** Elle ne s'exécute qu'une fois tes deux prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R1_26a, prediction_R1_26b]):
    print("⏳ Écris d'abord tes deux prédictions (des entiers), puis exécute à nouveau.")
else:
    print("round(3.46, 1)        =", round(3.46, 1))
    print("round(round(3.46, 1)) =", round(round(3.46, 1)))
    print("round(3.46)           =", round(3.46))

### Ex R1.30 — Les ordres de grandeur sur une échelle : de l'atome aux paramètres d'un LLM 🎨 ★ ⏱️ 15 min
**Objectif :** placer des nombres de toutes tailles sur une même échelle, en puissances de 10.  
**Prérequis :** fiche §6, « Ordres de grandeur » · **Parcours :** C

Sur une échelle ordinaire, $10^{-10}$ et $10^{12}$ ne tiennent pas sur le même dessin. On place plutôt l'**exposant** : chaque graduation est une puissance de 10, et un pas vers la droite multiplie par 10.

Complète `plot_magnitudes` : pour chaque nombre du dictionnaire `quantities` (cellule suivante), place un point à l'abscisse $\log_{10}(\text{valeur})$ (`math.log10`, qui donne l'exposant, même non entier : `math.log10(7e-5)` vaut environ −4,15) et écris son nom à côté. Un titre, l'axe horizontal nommé « puissance de 10 », une graduation par entier de −10 à 12 (`ax.set_xticks(range(-10, 13))`).

Puis lis ta figure : combien de puissances de 10 séparent l'épaisseur d'un cheveu du nombre de paramètres du modèle ? Note ta réponse dans ta copie de `06_mes_reponses.md`, et compare ta figure à celle de `05_solutions.ipynb`.

$\log_{10}$ est le **logarithme décimal** : $\log_{10}(10^k) = k$. R6 l'étudiera ; ici, retiens seulement qu'il donne l'exposant.

In [ ]:
# Approximate values (orders of magnitude only): label -> value
quantities = {
    "diamètre d'un atome (m)": 1e-10,
    "épaisseur d'un cheveu (m)": 7e-5,
    "secondes dans une journée": 8.64e4,
    "secondes dans une vie de 80 ans": 2.5e9,
    "êtres humains sur Terre": 8e9,
    "neurones d'un cerveau humain": 8.6e10,
    "paramètres d'un grand modèle de langage": 1e12,
}

In [ ]:
def plot_magnitudes(quantities):
    """Draw each quantity as a point at x = log10(value), with its label; title, axis label, ticks."""
    raise NotImplementedError("place les points, puis écris leurs noms")

In [ ]:
with wbm.attempt("R1.30"):
    plot_magnitudes(quantities)

## Partie C · Pourcentages et moyennes

Fiche §7 et §8. Tes deux dernières fonctions de `arith.py`, une expérience sur les hausses répétées, et une docstring.

### Ex R1.24 — Évolutions successives et moyenne pondérée · `successive_changes`, `weighted_mean` 🔨 ★ ⏱️ 30 min
**Objectif :** programmer le taux global d'évolutions successives et la moyenne pondérée, avec leurs cas d'erreur.  
**Prérequis :** fiche §7 et §8 ; workbookIA 0A §100.6.2 (`raise`) · **mymath :** `arith.py` · **Parcours :** C, E

Dans `mon_travail/mymath/arith.py` :
- a) `successive_changes(rates)` : les taux sont écrits en décimal (`0.10` pour +10 %). Multiplie les coefficients $1 + t$, puis repasse du coefficient global au taux global. Une liste vide donne `0.0` ; un taux inférieur à −1 (−100 %) lève une `ValueError`.
- b) `weighted_mean(values, weights)` : la somme des valeur × poids, divisée par la somme des poids. Trois cas lèvent une `ValueError` : des longueurs différentes, un poids négatif, des poids de somme nulle.

La vérification calcule `successive_changes([0.3, -0.2, 0.05])` et `weighted_mean([11, 17, 8], [2, 3, 5])`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R1")  # reload your saved file

In [ ]:
with wbm.attempt("R1.24a"):
    wbm.check("R1.24a", mymath.arith.successive_changes([0.3, -0.2, 0.05]), computed=True)
with wbm.attempt("R1.24b"):
    wbm.check("R1.24b", mymath.arith.weighted_mean([11, 17, 8], [2, 3, 5]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r1_arith.py", "test_successive_changes_ or test_weighted_mean_")

### Ex R1.27 — Après n hausses de 10 % : tracer et comparer à une hausse de 10n % 🔬 ★★ ⏱️ 20 min
**Objectif :** voir qu'une évolution répétée suit une puissance, qui finit par écraser une croissance de taux additionnés.  
**Prérequis :** fiche §7, « Évolutions successives » · **Parcours :** C

Un prix de 1 € augmente de 10 % chaque année. Après $n$ années, il vaut $1{,}1^n$ euros. Un camarade pense qu'il vaut $1 + 0{,}1n$ (« 10 % par an, donc $10n$ % en $n$ ans »).

- Trace les deux courbes, $1{,}1^n$ et $1 + 0{,}1n$, pour $n$ entier de 0 à 30 (points reliés, une légende, un titre, des axes nommés).
- a) Au bout de combien d'années le prix a-t-il **doublé** pour la première fois ($1{,}1^n \ge 2$) ? Un entier.
- b) Que vaut $1{,}1^{30}$, arrondi au centième ?

Pour a), une boucle `while` qui multiplie par 1,1 tant que le prix est inférieur à 2 suffit.

In [ ]:
def plot_growth(n_max=30):
    """Plot 1.1**n and 1 + 0.1*n for n = 0, 1, ..., n_max (legend, title, axis labels)."""
    raise NotImplementedError("trace les deux courbes")


answer_R1_27a = ...  # a) an integer (computed with a loop)
answer_R1_27b = ...  # b) a number rounded to 2 decimals

In [ ]:
with wbm.attempt("R1.27 plot"):
    plot_growth()
wbm.check("R1.27a", answer_R1_27a)
wbm.check("R1.27b", answer_R1_27b)

### Ex R1.31 — Écrire la docstring d'une fonction (format NumPy) 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire une docstring complète, au format NumPy, avec des exemples exécutables.  
**Prérequis :** fiche §8 (moyenne harmonique) ; workbookIA 0A §100.5.5 (docstrings) · **Parcours :** C

La fonction `harmonic_mean` (fiche §8, la moyenne harmonique) marche, mais personne ne sait comment s'en servir : elle n'a pas de docstring. Écris-la, **en anglais** comme tout le code du dépôt, au format NumPy que tu as vu dans 0A (§100.5.5) et dans les fichiers de `mymath` :
- une première ligne qui dit ce que fait la fonction ;
- `Parameters`, `Returns` et `Raises` (les deux cas d'erreur du code) ;
- `Examples` : au moins deux exemples `>>>` avec leur résultat, que tu calcules à la main (par exemple la moyenne harmonique de 90 et de 60 de la fiche). Écris le résultat exactement comme Python l'affiche : un flottant, `72.0`. Choisis des exemples dont le calcul en flottants tombe juste (évite les tiers et les septièmes).

La vérification regarde que les quatre sections sont là, puis **exécute tes exemples** (`doctest`) : un exemple dont le résultat diffère de ce que tu as écrit est signalé. Compare ensuite ta docstring à celle de `05_solutions.ipynb` : plusieurs rédactions sont justes.

In [ ]:
def harmonic_mean(values):
    """TODO: write the docstring (NumPy format)."""
    values = list(values)
    if len(values) == 0:
        raise ValueError("cannot compute the harmonic mean of an empty sequence")
    if any(v <= 0 for v in values):
        raise ValueError("the harmonic mean needs positive values")
    return len(values) / sum(1 / v for v in values)

In [ ]:
with wbm.attempt("R1.31"):
    doc = harmonic_mean.__doc__ or ""
    missing = [section for section in ("Parameters", "Returns", "Raises", "Examples") if section not in doc]
    if missing:
        print("❌ Ex R1.31 : il manque ces sections dans la docstring :", ", ".join(missing))
    else:
        runner = doctest.DocTestRunner()
        for test in doctest.DocTestFinder().find(harmonic_mean, "harmonic_mean", globs={"harmonic_mean": harmonic_mean}):
            runner.run(test)
        failed, attempted = runner.summarize(verbose=False)
        if attempted < 2:
            print(f"❌ Ex R1.31 : {attempted} exemple(s) >>> dans la docstring ; il en faut au moins deux.")
        elif failed:
            print(f"❌ Ex R1.31 : {failed} exemple(s) sur {attempted} ne donnent pas le résultat écrit (détail au-dessus).")
        else:
            print(f"✅ Ex R1.31 : quatre sections, et {attempted} exemples qui donnent bien le résultat écrit.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu sans erreur avec des priorités, des négatifs, des fractions et des puissances, et repères-tu les « règles » qui n'existent pas ?
2. Sais-tu écrire un nombre en notation scientifique, l'arrondir et donner son ordre de grandeur ?
3. Sais-tu enchaîner des évolutions en pourcentage et calculer une moyenne pondérée, à la main et avec `mymath` ?

**La suite** : R2, *Calcul littéral et équations*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.